In [ ]:
import numpy as onp
import matplotlib.pyplot as plt
from jax import numpy as np
from vmc_core import *
# from vmc_core import RNG

In [2]:
# for nu, nd in [(1,1), (2,2), (3,3), (6,6)]:
#     print(f"N_up={nu}, N_down={nd} | orbitals={occupied_orbitals(nu)} | E0={exact_energy(nu,nd)}")
# assert [exact_energy(n,n) for n in (1,2,3,6)] == [2,6,10,28]

In [3]:
# # Same-spin swap: one minus sign, no change in |psi|².
# X_test = RNG.normal(size=(20,6,2))
# X_swapped = X_test.copy()
# X_swapped[:,[0,1],:] = X_test[:,[1,0],:]
# a_test = RNG.normal(size=len(S_K))   # nonzero Jastrow: it must be symmetric under the swap too
# l1 = logpsi(X_test, a_test, 3)
# l2 = logpsi(X_swapped, a_test, 3)
# onp.testing.assert_allclose(l2, l1, atol=1e-12)
# s1 = np.linalg.slogdet(build_slaters(X_test, 3)[0])[0]
# s2 = np.linalg.slogdet(build_slaters(X_swapped, 3)[0])[0]
# onp.testing.assert_array_equal(s2, -s1)
# print("Same-spin exchange: sign flips and |psi|² is unchanged.")

In [4]:
# sweep_step = 5   # Metropolis steps between recorded samples: set it to about the tau printed below
# n_samples = 5000 # recorded samples per chain
# runs = {}        # (nu,nd) -> dictionary of results returned by sample_mcmc
# autocorr = {}
# # for nu,nd in [(1,1),(2,2),(3,1)]:
# for nu,nd in [(4,4)]:
#     res = sample_mcmc(logpsi, n_particles=nu + nd, n_samples=n_samples, n_discard=400, step_size=1.1, n_up=nu, n_chains=200,
#                       sweep_step=sweep_step, measures=("energy", "V", "density", "acceptance"))
#     runs[(nu,nd)] = res
#     e, err, sigma = res["energy"]["mean"], res["energy"]["error"], res["energy"]["sigma"]
#     V = res["V"]
#     expected = exact_energy(nu,nd)
#     print(f'{nu}+{nd}: VMC E={e[-1]:.12f} ± {err[-1]:.2g} | exact={expected} | '
#           f'acceptance={res["acceptance"]:.3f} | std of E_L={sigma[-1]:.2g}')
#     assert onp.abs(e[-1]-expected)<1e-10

#     autocorr[(nu,nd)] = autocorrelation(V)
#     tau = autocorrelation_time(autocorr[(nu,nd)])
#     print(f'{nu}+{nd}: autocorrelation time tau = {tau:.1f} recorded samples '
#           f'(sweep_step={sweep_step}) -> effective samples per chain ~ {len(V)/tau:.0f}')

In [5]:
# # Running mean vs iteration
# # E_L is constant for an exact eigenstate, so its running mean sits on E from the start and does not test the sampler.
# # Virial theorem (harmonic trap): <V> = <T> = E/2, so 2<V> -> E only if the samples are really distributed as |psi|^2
# fig, ax = plt.subplots(figsize=(6.6,3.7))
# for k, ((nu,nd), res) in enumerate(runs.items()):
#     V = res["V"]
#     it = onp.arange(1, len(V) + 1)
#     # running mean of each chain separately; the chains are independent, so the spread between them
#     # gives an honest error bar even though samples inside one chain are correlated
#     run_chain = onp.cumsum(V, axis=0) / it[:, None]             # shape (n_samples, n_chains)
#     mean = run_chain.mean(axis=1)
#     err = run_chain.std(axis=1, ddof=1) / onp.sqrt(run_chain.shape[1])
#     ax.plot(it, mean, color=f'C{k}', label=f'{nu}+{nd}: running mean of $V$')
#     ax.fill_between(it, mean - err, mean + err, color=f'C{k}', alpha=0.25, lw=0)
#     ax.axhline(exact_energy(nu,nd)/2, color=f'C{k}', ls='--', lw=0.8)
# ax.set(xlabel='Iteration (recorded sample)', ylabel='Running mean', title='Convergence to the expec. val. of V (dashed)')
# ax.legend(frameon=False, fontsize=8)
# plt.show()

In [6]:
# # Radial density: VMC histogram (error bars from the spread of the chains) and exact n(r), both integrate to N
# # n_exact is the angular average for the 4+4 filling, so it is only drawn for that one
# fig, ax = plt.subplots(figsize=(6.6,3.7))
# for k, ((nu,nd), res) in enumerate(runs.items()):
#     r, density, err = density_from_hist(res["density"]["hist"], res["density"]["edges"], n_samples)
#     ax.errorbar(r, density, yerr=err, fmt='.', ms=5, lw=0.8, color=f'C{k}', label=f'{nu}+{nd}: VMC')
#     if (nu,nd) == (4,4):
#         ax.plot(r, n_exact(r), color='k', lw=1, zorder=3, label=f'{nu}+{nd}: exact')
# ax.set(xlabel='$r$', ylabel='$n(r)$ (particles per unit area)', title='Radial density')
# ax.legend(frameon=False, fontsize=8)
# plt.show()